# Day 83 — Optimization

This notebook covers Reverse-Mode Automatic Differentiation, Vector-Jacobian Products (VJPs), Directed Acyclic Graph (`DAG`) Traversal, Gradient Accumulation Mechanics (`+=`), and Evaluation Context Hygiene (`torch.no_grad()`)

## Validation goals
- Formulate why `.backward()` computes Vector-Jacobian Products (VJPs) and expects a scalar starting objective with an implicit upstream gradient of `1.0`
- Trace reverse-topological traversal across `grad_fn` operation nodes and multivariable branching paths in a dynamic computation graph
- Analyze how un-zeroed gradient accumulation (`+=`) distorts update directions and corrupts momentum/adaptive optimizer statistics
- Reinforce evaluation loop hygiene by disabling graph construction and memory overhead inside `with torch.no_grad():`

### 1. How Autograd and the Computation Graph Work

When a tensor has `requires_grad=True`, PyTorch tracks every operation performed on it to build a Directed Acyclic Graph (DAG) on the fly:
- Leaf Tensors: Tensors created directly by you (such as model weights, biases, or a parameter tensor with requires_grad=True) that are not the result of an operation. Their gradients accumulate in `.grad`.
- Non-Leaf Intermediate Tensors: Outputs of intermediate operations (like z = w * x + b). By default, PyTorch frees their intermediate gradient buffers after backward passes to save memory.
- `.backward()`: Starting from a scalar output tensor (such as a loss value), `.backward()` walks backward through the graph using the chain rule and populates .grad on leaf tensors with `requires_grad=True`.

### 2. Gradients in Action: A Concrete Example

Let's trace a small differentiable expression where we can verify the math by hand:$$y = 3w^2 + 2w + 5$$Taking the derivative with respect to $w$:$$\frac{dy}{dw} = 6w + 2$$At $w = 4.0$, the exact derivative is $6(4.0) + 2 = 26.0$. Let's watch PyTorch compute this automatically:

In [1]:
import torch

# Create a leaf parameter tensor with gradient tracking enabled
w = torch.tensor(4.0, dtype=torch.float32, requires_grad=True)

# Forward computation builds the autograd graph dynamically
y = 3.0 * (w ** 2) + 2.0 * w + 5.0

# Compute gradients of scalar y with respect to w
y.backward()

print("--- Day 82: Autograd Derivative Check ---")
print(f"w value:       {w.item():.2f}")
print(f"y value:       {y.item():.2f}")
print(f"dy/dw (.grad): {w.grad.item():.2f}")
print(f"Is leaf?       {w.is_leaf}")

--- Day 82: Autograd Derivative Check ---
w value:       4.00
y value:       61.00
dy/dw (.grad): 26.00
Is leaf?       True


### 3. Gradient Accumulation and Zeroing Gradients

A critical rule in PyTorch autograd: `.grad` accumulates (adds) across calls rather than overwriting.Why? In architectures like Recurrent Neural Networks or gradient accumulation across micro-batches, summing gradients across steps is useful. In standard mini-batch training loops, however, forgetting to clear old gradients causes stale gradients from step $t-1$ to leak into step $t$.

In [2]:
import torch

x = torch.tensor(2.0, dtype=torch.float32)
w = torch.tensor(3.0, dtype=torch.float32, requires_grad=True)

# Step 1: loss = (w * x - 10)^2 -> d(loss)/dw = 2 * (6 - 10) * 2 = -16
loss1 = (w * x - 10.0) ** 2
loss1.backward()
print(f"After 1st backward, w.grad = {w.grad.item():.1f}")

# Step 2 WITHOUT zeroing: w.grad ADDS the new gradient (-16) to the existing (-16)
loss2 = (w * x - 10.0) ** 2
loss2.backward()
print(f"After 2nd backward (un-zeroed), w.grad = {w.grad.item():.1f}")

# Reset gradient before computing a fresh gradient
w.grad.zero_()
loss3 = (w * x - 10.0) ** 2
loss3.backward()
print(f"After zeroing + 3rd backward, w.grad = {w.grad.item():.1f}")

After 1st backward, w.grad = -16.0
After 2nd backward (un-zeroed), w.grad = -32.0
After zeroing + 3rd backward, w.grad = -16.0


### 4. Disabling Gradient Tracking: `torch.no_grad()`

Building and storing a computation graph costs memory and compute. During inference, validation, or test evaluation, you are not calling `.backward()`, so you do not need graph tracking.

Wrapping evaluation or inference blocks in with `torch.no_grad()`: disables gradient recording inside that block, cutting memory overhead and speeding up execution:

### Guided Question:

Why does `.backward()` require the starting tensor (such as loss) to be a scalar in standard usage, and how does autograd apply the chain rule across operations?

### Answer:

**1. Why .backward() Expects a Scalar**

In automatic differentiation (such as PyTorch's autograd), .backward() computes vector-Jacobian products (VJPs) rather than a full Jacobian matrix.

Suppose a function maps an input vector $\mathbf{x} \in \mathbb{R}^n$ to an output tensor $\mathbf{y} \in \mathbb{R}^m$. Its Jacobian matrix $\mathbf{J}$ has shape $m \times n$:$$\mathbf{J}_{ij} = \frac{\partial y_i}{\partial x_j}$$

Materializing the full Jacobian for large tensors is prohibitively expensive in memory and compute. Instead, reverse-mode autodiff propagates an upstream gradient vector $\mathbf{v}$ backward from the output to compute $\mathbf{v}^T \mathbf{J}$:$$\left(\mathbf{v}^T \mathbf{J}\right)_j = \sum_{i=1}^{m} v_i \frac{\partial y_i}{\partial x_j}$$

- When the output is a scalar ($m = 1$): There is only one output element ($y_1$, typically the scalar loss), so its gradient with respect to itself is simply $1$. Autograd automatically supplies an implicit upstream gradient of 1.0 ($\mathbf{v} = [1]$), giving the exact gradient $\frac{\partial \text{loss}}{\partial \mathbf{x}}$.
- When the output is non-scalar ($m > 1$): There is no single scalar objective being differentiated, and the upstream gradient $\mathbf{v}$ is ambiguous. That is why `.backward()` raises an error unless you either reduce the tensor to a scalar (e.g., `.sum()` or `.mean()`) or explicitly pass gradient ($\mathbf{v}$) matching the tensor's shape.

**2. How Autograd Applies the Chain Rule Across Operations**

Autograd records operations in a directed acyclic graph (DAG) during the forward pass:
- Leaf tensors are inputs or model parameters (like weights $\mathbf{w}$) with requires_grad=True.
- Function nodes (grad_fn) represent differentiable operations (addition, matrix multiplication, `ReLU`, etc.) connecting input tensors to output tensors.

During `.backward()`, autograd traverses the graph in reverse topological order from the output back to the leaves:
- Initialize: Seed the starting tensor's incoming gradient with 1.0 (for a scalar) or the provided gradient tensor.
- Step backward through each operation: For an operation $z = f(\mathbf{u}, \mathbf{w})$ that has already received $\frac{\partial L}{\partial z}$ from downstream nodes, its backward rule computes how $\frac{\partial L}{\partial z}$ flows to each input using that operation's local partial derivatives:$$\frac{\partial L}{\partial \mathbf{u}} = \left(\frac{\partial L}{\partial z}\right) \cdot \frac{\partial z}{\partial \mathbf{u}}, \qquad \frac{\partial L}{\partial \mathbf{w}} = \left(\frac{\partial L}{\partial z}\right) \cdot \frac{\partial z}{\partial \mathbf{w}}$$

(where the exact contraction/dot product depends on tensor shapes and the specific op's VJP formula).

Accumulate gradients: If a tensor is used in multiple branches of the graph, its contributions from each branch are summed (multivariable chain rule: $\frac{\partial L}{\partial x} = \sum_k \frac{\partial L}{\partial z_k}\frac{\partial z_k}{\partial x}$). At leaf tensors with requires_grad=True, these gradients accumulate into .grad.

### Guided Question:

What happens if you call `loss.backward()` on consecutive training iterations without clearing previous gradients, and why is that a bug in standard training loops?

### Answer:

**What Happens: Gradients Accumulate (`+=`)**

By default, autograd accumulates gradients into a parameter's .grad attribute rather than overwriting them (param.grad += new_grad).

On iteration $t$, if you don't zero or clear prior gradients first:$$\text{param.grad}^{(t)} = \text{param.grad}^{(t-1)} + \frac{\partial \mathcal{L}^{(t)}}{\partial \text{param}}$$

Instead of representing the derivative of the current batch's loss $\mathcal{L}^{(t)}$, .grad becomes the sum of gradients across all prior steps since the last zeroing operation (such as `optimizer.zero_grad()` or `param.grad.zero_()`).

Why It Is a Bug in Standard Training LoopsIn a standard mini-batch gradient descent loop, each step is meant to estimate the update direction from the current batch alone:
- Wrong update direction and magnitude: Adding stale gradients from earlier batches distorts the descent direction and inflates step sizes—often causing noisy updates, oscillation, or training instability/divergence.
- Momentum and adaptive optimizers compound the error: Optimizers like SGD with momentum, Adam, or AdamW rely on `.grad` at each step to update moving averages (first/second moments or velocity). Feeding accumulated multi-step gradients corrupts those internal statistics.
- Silent failure mode: Unlike a shape mismatch or missing tensor, forgetting to clear gradients usually doesn't crash your code—it quietly degrades convergence and model quality, making it subtle to debug.

(Note: Intentional gradient accumulation across micro-batches is a deliberate exception where you want to sum gradients over $k$ steps before calling `optimizer.step()` and zeroing—typically scaling each micro-batch loss by $1/k$.)

### Guided Question:

Why should validation and test loops run inside with `torch.no_grad():` instead of tracking gradients like during training?

### Answer:

Running validation and test loops inside with `torch.no_grad():` disables autograd gradient tracking during evaluation, where you only need forward predictions and metrics—never parameter updates.

**1. Lower Memory Usage (No Computation Graph Saved)**

During training, PyTorch builds a computation graph and saves intermediate activations/tensors needed later for `.backward()`. In validation and testing, no backward pass ever happens, so keeping that graph around is pure overhead. Wrapping evaluation in `torch.no_grad()` avoids constructing the graph and frees intermediate buffers much sooner, reducing peak GPU/CPU memory and lowering the risk of out-of-memory errors on larger evaluation batches.

**2. Faster Execution**

Disabling gradient tracking skips bookkeeping for `requires_grad`, creating `grad_fn` nodes, and recording graph edges on every operation. This reduces CPU/Python dispatch overhead and lets operations run faster, speeding up evaluation passes across large validation or test sets.

**3. Prevents Accidental Gradient Updates or Side Effects**

During evaluation, you should never call `.backward()` or `optimizer.step()`. Running inside `torch.no_grad()` makes it impossible to accidentally accumulate gradients into .grad or modify model parameters during metrics calculation.